# 01 — Exploratory Data Analysis

Proyecto Final MLOps · Universidad de Medellín · WorldCup 2026

Objetivo: entender la historia internacional de partidos de selecciones
(martj42) y el histórico de Mundiales FIFA 1930-2022 (Fjelstul). Identificar
sesgos, distribuciones y correlaciones clave antes de entrenar.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
from worldcup2026.data.ingest import ingest_all
from worldcup2026.data.preprocess import build_match_history

pd.set_option('display.max_columns', 40)

In [ ]:
paths = ingest_all()
paths

In [ ]:
matches = build_match_history()
print(matches.shape)
matches.head()

## Distribución temporal

In [ ]:
matches['year'] = matches['date'].dt.year
ax = matches.groupby('year').size().plot(figsize=(12,3), title='Partidos internacionales por año')
ax.set_xlabel('Año'); ax.set_ylabel('# partidos')

## Resultados — distribución de goles

In [ ]:
matches[['home_score','away_score']].describe()

In [ ]:
result = (matches['home_score'] > matches['away_score']).map({True:'home_win', False:'not'})
result[matches['home_score'] == matches['away_score']] = 'draw'
result[matches['home_score'] < matches['away_score']] = 'away_win'
result.value_counts(normalize=True).plot(kind='bar', title='Proporción de resultados (toda la historia)')

## Ventaja de local vs. cancha neutral

In [ ]:
def win_rate(df):
    return (df['home_score'] > df['away_score']).mean()

print('Win rate local (toda cancha):', win_rate(matches).round(3))
print('Win rate local (NO neutral):', win_rate(matches[~matches['neutral']]).round(3))
print('Win rate local (neutral):   ', win_rate(matches[matches['neutral']]).round(3))

## Conclusiones clave

- El dataset cubre ~150 años y ~49k partidos, suficiente para un Elo robusto.
- Ventaja de local es clara: ~10-15 puntos porcentuales vs. neutral.
- La clase 'home_win' domina — usamos `f1_macro` y `log_loss` en vez de accuracy.
- Los torneos WC + continentales son minoría — se ponderan con K más alto en el Elo.